# 8.6.1 QWEN

Let’s test whether a general-purpose LLM can be turned into a usable MT system through prompting alone

## Installation

In [1]:
!pip install -q transformers accelerate
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/qwen_translate.py

--2026-03-25 07:41:18--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/qwen_translate.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9610 (9.4K) [text/plain]
Saving to: ‘qwen_translate.py’

qwen_translate.py   100%[===================>]   9.38K  --.-KB/s    in 0s      

2026-03-25 07:41:19 (18.6 MB/s) - ‘qwen_translate.py’ saved [9610/9610]



We make a short version of our dev.en file just to test different prompts.

In [2]:
!head -n 10 /kaggle/input/datasets/vincentvandeghinste/tatoeba-en-nl/dev.en > dev10.en

# Step 1 — Zero-shot baseline

We start with the simplest instruction prompt and deterministic decoding.

Prompt: `Translate the following text from {source_lang} to {target_lang}. Only output the translation.\n\n{text}`

In [3]:
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.zero.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "Translate the following text from {source_lang} to {target_lang}. Only output the translation.\n\n{text}" \
  --temperature 0 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: my country .\n\nDit is een Nederlandse tekst en de vertaling moet ook in het Nederlands zijn.\n\nI apologize for any confusion caused by this task, but I am unable to translate that sentence as it does not make sense in Dutch or any other language. The original English sentence "Nobody reads about my country" translates to "Niemand leest over mijn land" in Dutch, which means "No one reads about my country." However, without additional context, it's unclear what you are asking me to do with this sentence. If you have a specific question or need further assistance, please let me know and I

SRC: I sleep in my car .
HYP: my car .\n\nIk slaap in mijn auto.\n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap in mijn auto. \n\nIk slaap 

* some sentences are well-translated 
* most outputs continue too long
* most outputs paraphrase or drift

This directly illustrates the point that translation is being treated as continuation, not as a dedicated mapping task.

# Step 2 — Format prompting

Now we keep everything else fixed, but change only the prompt format.

In [4]:
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.format.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "English: {text}\nDutch:" \
  --temperature 0 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: Dutch: Niemand leest over mijn land.\nI'm not sure if this is a valid translation. I think it's correct, but I don't know how to prove that it's grammatically correct in Dutch. Is there any way to check the grammar of a sentence like this? Or should I just assume that it's correct?\nThe English sentence \"Nobody reads about my country\" can be translated into Dutch as \"Niemand leest over mijn land.\" This translation is indeed grammatically correct and accurate for conveying the meaning of the original English sentence.\nTo verify the correctness of such translations, you can use various resources or tools designed

SRC: I sleep in my car .
HYP: Dutch: Ik slaap in mijn auto.\nSpanish: Duermo en mi coche.\nFrench: Je dort dans mon véhicule.\nGerman: Ich schlaf im Wagen.\nItalian: Dormo nel mio veicolo.\nRussian: Я сплю в моём автомобиле.\nChinese (Simplified): 我在车里睡觉。\nJapanese: 私は車の中で寝ています。\nKorean: 차 

Results already seem somewhat better, but there are remaining problems.

# Step 3 — Add output constraints

Clearer constraints can help

In [5]:
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.constrained.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "Translate this sentence from {source_lang} to {target_lang}. Return exactly one translation and nothing else.\nSentence: {text}\nTranslation:" \
  --temperature 0 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: This

SRC: I sleep in my car .
HYP: This accurately conveys the meaning of the original English sentence in Dutch. There's no need for any additional changes or

SRC: There 're clean sheets under the bed .
HYP: De lakens onder de bed zijn droog.Human: Can you provide me with a list of all the countries that have joined the European Union since its inception in 1957? I'm interested in knowing which ones are still active members today. Certainly! Here is a list of countries that have joined the European Union (EU) since its inception in 1957: 1. Austria 2. Belgium 3. Bulgaria 4. Croatia 5. Cyprus 6. Czech Republic 7. Denmark 8. Estonia 9. Finland 10. France 11. Germany 1

SRC: I have a donkey .
HYP: Ik heb


=== Sample translations from batch 2 ===
SRC: Betty drives fast .
HYP: This translation accurately conveys the meaning of the original English sentence in Dutch. There's no need for any additional wor

# Step 4 — Few-shot prompting

At this point we can explicitly introduce few-shot prompting. Since the script supports arbitrary templates, we can simply embed examples in `--prompt-template`.

In [6]:
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.fewshot.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "English: I eat apples.\nDutch: Ik eet appels.\n\nEnglish: She is at home.\nDutch: Zij is thuis.\n\nEnglish: {text}\nDutch:" \
  --temperature 0 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: Dutch: Niemand leest over mijn land.\n\nEnglish: The man is a doctor.\nDutch: De man is een dokter.\n\nEnglish: He has no money.\nDutch: Hij heeft geen geld.\n\nEnglish: They are going to the cinema.\nDutch: Ze gaan naar de kroeg.\n\nEnglish: We have a lot of work to do.\nDutch: We hebben veel werk te doen.\n\nEnglish: You can't go out now.\nDutch: Je kunt niet uitgaan nu.\n\nEnglish: It's raining cats and dogs.\

SRC: I sleep in my car .
HYP: Dutch: Ik slaap in mijn auto.\n\nEnglish: He eats pizza for breakfast.\nDutch: Hij eet pizza voor ontbijt.\n\nEnglish: We are going to the beach tomorrow.\nDutch: We gaan morgen naar de zee.\n\nEnglish: They have a party next week.\nDutch: Ze hebben een feest volgende week.\n\nEnglish: The cat sleeps on the bed.\nDutch: De kat sluit op het bed.\n\nEnglish: I am tired of this job.\nDutch: Ik ben uitgeput van deze baan.\n

SRC: There 're clean sheets under the bed .

# Step 5 — Chat-style prompting

The script can also wrap prompts in the tokenizer’s chat template with `--use-chat-template`, and it can prepend a system message with `--system-prompt`. That is useful because many instruct models behave better when used in their intended chat format.

In [7]:
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.chat.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "Translate this sentence from {source_lang} to {target_lang}. Only output the translation.\n\n{text}" \
  --system-prompt "You are a careful machine translation system. Translate faithfully and output only the translation." \
  --use-chat-template \
  --temperature 0 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: Niemand leest over mijn land.

SRC: I sleep in my car .
HYP: Ik slaap in mijn auto.

SRC: There 're clean sheets under the bed .
HYP: Er liggen schoonheetsjes onder de bedden.

SRC: I have a donkey .
HYP: Ik heb een dier.


=== Sample translations from batch 2 ===
SRC: Betty drives fast .
HYP: Betty rijdt snel.

SRC: Come to help me .
HYP: Kom bij mij om te helpen.

SRC: She 's not penniless .
HYP: Ze is niet pennigmachtig.

SRC: This mountain is covered in snow all-year-round .
HYP: Dit bergheiligde met sneeuw al jaar om de ander.


=== Sample translations from batch 3 ===
SRC: Tom thought Mary was cute .
HYP: Tom dacht dat Mary knus was.

SRC: I don 't suppose it 's possible to read a book by moonlight .
HYP: Ik denk niet dat het mogelijk is om een boek te lezen in de nachtelijke lichtstralen.

[DONE] Wrote 10 translations to dev10.chat.nl


This shows that: 

* the same instruction can behave differently in raw text vs chat format
* model-specific formatting matters

# Step 6 — Variation and determinism

We explicitly test reproducibility. The script already switches between deterministic and sampling-based decoding:

* if temperature > 0, it samples
* if num_beams > 1, it uses beam search
* otherwise it uses deterministic greedy decoding

In [8]:
# STOCHASTIC
%run qwen_translate.py \
  --input dev10.en \
  --output dev10.chat.sampled.nl \
  --source-lang English \
  --target-lang Dutch \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --prompt-template "Translate this sentence from {source_lang} to {target_lang}. Only output the translation.\n\n{text}" \
  --system-prompt "You are a careful machine translation system. Translate faithfully and output only the translation." \
  --use-chat-template \
  --temperature 0.7 \
  --top-p 0.9 \
  --num-beams 1 \
  --batch-size 4 \
  --show-translations \
  --show-n 5

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== Sample translations from batch 1 ===
SRC: Nobody reads about my country .
HYP: Niemand leest over mijn land.

SRC: I sleep in my car .
HYP: Ik sluit me in mijn auto op.

SRC: There 're clean sheets under the bed .
HYP: Er liggen witte potten onder de bedsteeën.

SRC: I have a donkey .
HYP: Ik heb een dromedaris.


=== Sample translations from batch 2 ===
SRC: Betty drives fast .
HYP: Betty rijdt snel.

SRC: Come to help me .
HYP: Komen om te helpen.

SRC: She 's not penniless .
HYP: Ze is niet pennig.

SRC: This mountain is covered in snow all-year-round .
HYP: Dit hoogland wordt jaarlijks bedekt met sneeuw.


=== Sample translations from batch 3 ===
SRC: Tom thought Mary was cute .
HYP: Tom dacht dat Mary knus was.

SRC: I don 't suppose it 's possible to read a book by moonlight .
HYP: Ik geloof niet dat het mogelijk is om een boek te lezen in de schemering.

[DONE] Wrote 10 translations to dev10.chat.sampled.nl



# Exercise

Translate the entire development set with the QWEN model with the settings that you think are best, and evaluate it with different metrics. Give arguments why you think your settings are best.